# PI4 — Municípios comparáveis a Guaratinguetá

Notebook preparatório da etapa **E02**. O objetivo é construir um **pool exploratório** de municípios paulistas estruturalmente semelhantes a Guaratinguetá, sem usar os próprios indicadores de infraestrutura como critério de semelhança.

A seleção final de comparáveis continua sendo uma decisão analítica posterior; este notebook não congela o grupo final.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from datetime import datetime
import subprocess, sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

PROJECT_NAME = 'UNIVESP — PI4 — Infraestrutura Escolar — Guaratinguetá'
candidates = [Path('/content/drive/MyDrive') / PROJECT_NAME, Path('/content/drive/My Drive') / PROJECT_NAME]
PROJECT_ROOT = next((p for p in candidates if p.exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Adicione um atalho da pasta compartilhada do PI4 ao Meu Drive.')
BASE_ANALITICA = PROJECT_ROOT / '01_Dados' / '2_tratamentos_dados' / 'base_analitica'


In [ ]:
!rm -rf /content/pi4_repo
!git clone -q --depth 1 https://github.com/felipecsr/univesp-projeto-integrador-4.git /content/pi4_repo
sys.path.insert(0, '/content/pi4_repo')

from src.validate_analytic_base import validate_execution
from src.eda import latest_execution_dir, load_materialized_panel, municipality_profiles, comparable_pool

REPO_COMMIT = subprocess.check_output(['git','-C','/content/pi4_repo','rev-parse','HEAD'], text=True).strip()
RUN_DIR = latest_execution_dir(BASE_ANALITICA)
print('Commit:', REPO_COMMIT)
print('Base usada:', RUN_DIR)


## Gate P04


In [ ]:
reconciliacao = validate_execution(RUN_DIR)
display(reconciliacao)
assert not (reconciliacao['status'] == 'FAIL').any(), 'P04 falhou. Corrija a base antes da EDA.'
print('GATE P04: PASS')


## Perfil de Guaratinguetá em 2025

O primeiro pool usa apenas características estruturais do sistema escolar: número de escolas, matrículas, composição por rede e participação rural. Indicadores de infraestrutura ficam de fora para não escolher municípios que já se parecem justamente no resultado que queremos comparar.


In [ ]:
panel = load_materialized_panel(RUN_DIR)
profiles = municipality_profiles(panel, year='2025')
target = profiles[profiles['CO_MUNICIPIO'].astype(str) == '3518404']
display(target)

quantis = profiles[['escolas','matriculas','mediana_matriculas_escola']].quantile([0.1,0.25,0.5,0.75,0.9]).reset_index().rename(columns={'index':'quantil'})
display(quantis)


## Sensibilidade do filtro de porte

Antes de olhar nomes, avaliamos quantos municípios entram com janelas de ±50%, ±40% e ±30% em escolas e matrículas.


In [ ]:
rows = []
for limite in [0.5, 0.4, 0.3]:
    pool_tmp = comparable_pool(
        panel,
        school_ratio=(1-limite, 1+limite),
        enrollment_ratio=(1-limite, 1+limite),
    )
    rows.append({'janela': f'±{int(limite*100)}%', 'municipios': len(pool_tmp)})
sensibilidade = pd.DataFrame(rows)
display(sensibilidade)


## Pool exploratório padrão

Usamos inicialmente a janela ±40%. A coluna `distancia_estrutural` serve apenas para ordenar a inspeção; **não é ainda uma escolha automática dos comparáveis finais**.


In [ ]:
pool = comparable_pool(panel, school_ratio=(0.6,1.4), enrollment_ratio=(0.6,1.4))
display(pool.head(25))
print('Municípios no pool:', len(pool))


In [ ]:
fig, ax = plt.subplots(figsize=(8,6))
ax.scatter(profiles['escolas'], profiles['matriculas'], alpha=0.35)
t = target.iloc[0]
ax.scatter([t['escolas']], [t['matriculas']], s=90, marker='x')
ax.annotate('Guaratinguetá', (t['escolas'], t['matriculas']), xytext=(6,6), textcoords='offset points')
ax.set_title('Municípios de SP — porte do sistema escolar em 2025')
ax.set_xlabel('Escolas ativas')
ax.set_ylabel('Matrículas')
ax.grid(alpha=0.2)
plt.show()


## Materialização da exploração de comparáveis


In [ ]:
EDA_ROOT = PROJECT_ROOT / '01_Dados' / '2_tratamentos_dados' / 'eda'
EDA_RUN = EDA_ROOT / f'comparaveis_{datetime.now().strftime("%Y%m%d_%H%M%S")}'
EDA_RUN.mkdir(parents=True, exist_ok=True)

profiles.to_csv(EDA_RUN / 'perfis_municipios_sp_2025.csv', index=False, encoding='utf-8')
pool.to_csv(EDA_RUN / 'pool_comparaveis_2025.csv', index=False, encoding='utf-8')
sensibilidade.to_csv(EDA_RUN / 'sensibilidade_filtro_comparaveis.csv', index=False, encoding='utf-8')
quantis.to_csv(EDA_RUN / 'quantis_perfis_municipios.csv', index=False, encoding='utf-8')
reconciliacao.to_csv(EDA_RUN / 'gate_p04.csv', index=False, encoding='utf-8')

manifest = pd.DataFrame([{
    'executado_em': datetime.now().isoformat(timespec='seconds'),
    'repo_commit': REPO_COMMIT,
    'base_origem': str(RUN_DIR),
    'municipios_sp': len(profiles),
    'municipios_pool_padrao': len(pool),
    'criterio_padrao': '±40% escolas e matrículas + ordenação estrutural por rede/zona',
}])
manifest.to_csv(EDA_RUN / 'manifesto_comparaveis.csv', index=False, encoding='utf-8')
display(manifest)
print('Exploração gravada em:', EDA_RUN)


## Evidências manuais de E02

Preserve o notebook com outputs e registre: **(1)** perfil de Guaratinguetá + quantis; **(2)** sensibilidade das janelas; **(3)** primeiras linhas do pool; **(4)** dispersão escolas × matrículas. Depois, deixe o card `E02M` em `Revisão`.
